In [ ]:
import numpy as np
import json, os
from scipy import stats, optimize
from pathlib import Path
from collections import Counter

from google.colab import drive
drive.mount('/content/drive')

RESULTS_BASE = Path('/content/drive/MyDrive/LRTIA/Results')
print(f'Mounted: {RESULTS_BASE.exists()}')

In [ ]:
# === Load all cached results (Llama + Mistral) ===
common_x = np.arange(1, 101)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def get_binned(corrected_marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = corrected_marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    return np.array(bc), np.array(bm)

# Dataset definitions: (key, label, intact_path, shuffled_path, filter_pop)
DATASETS = []

# Llama crosslingual
llama_dir = RESULTS_BASE / 'Llama_crosslingual'
for code, name in [('zh','Chinese'), ('ja','Japanese'), ('ko','Korean'),
                    ('tr','Turkish'), ('ar','Arabic'), ('fi','Finnish')]:
    ip = llama_dir / f'wiki_{code}_intact.json'
    sp = llama_dir / f'wiki_{code}_shuffled.json'
    if ip.exists():
        DATASETS.append((f'Llama_{name}_Wiki', f'{name} Wiki (Llama)', ip, sp, None))

for key, label in [('buckeye','Buckeye (Llama)'), ('french','French spoken (Llama)')]:
    ip = llama_dir / f'{key}_intact.json'
    sp = llama_dir / f'{key}_shuffled.json'
    if ip.exists():
        DATASETS.append((f'Llama_{key}', label, ip, sp, None))

# Mistral crosslingual
mistral_wiki = RESULTS_BASE / 'Wiki_multilingual_finegrain'
for code, name in [('zh','Chinese'), ('ja','Japanese'), ('ko','Korean'),
                    ('tr','Turkish'), ('ar','Arabic'), ('fi','Finnish')]:
    ip = mistral_wiki / f'{code}_intact_v1.json'
    sp = mistral_wiki / f'{code}_shuffled_v1.json'
    if ip.exists():
        DATASETS.append((f'Mistral_{name}_Wiki', f'{name} Wiki (Mistral)', ip, sp, None))

bk_ip = RESULTS_BASE / 'Buckeye_finegrain' / 'buckeye_intact_v1.json'
bk_sp = RESULTS_BASE / 'Buckeye_finegrain' / 'buckeye_shuffled_v1.json'
if bk_ip.exists():
    DATASETS.append(('Mistral_Buckeye', 'Buckeye (Mistral)', bk_ip, bk_sp, None))

fr_ip = RESULTS_BASE / 'French_oral_finegrain' / 'french_oral_intact_v1.json'
fr_sp = RESULTS_BASE / 'French_oral_finegrain' / 'french_oral_shuffled_v1.json'
if fr_ip.exists():
    DATASETS.append(('Mistral_French', 'French spoken (Mistral)', fr_ip, fr_sp, None))

# Load all
all_data = {}
for key, label, ip, sp, fpop in DATASETS:
    with open(ip) as f:
        intact = json.load(f)
    with open(sp) as f:
        shuffled = json.load(f)
    if fpop:
        intact = [c for c in intact if c.get('population') == fpop]
        shuffled = [c for c in shuffled if c.get('population') == fpop]
    if len(intact) < 5 or len(shuffled) < 5:
        continue
    ipc = compute_raw_ppl_curve(intact)
    spc = compute_raw_ppl_curve(shuffled)
    corr = -np.diff(ipc) - (-np.diff(spc))
    bc, bm = get_binned(corr)
    if len(bc) >= 4:
        all_data[key] = {'label': label, 'bc': bc, 'bm': bm, 'n': len(intact)}

print(f'Loaded {len(all_data)} datasets')
for k, v in all_data.items():
    print(f'  {v["label"]} (n={v["n"]})')

In [ ]:
# === Fit all functional forms ===
def power_law(x, a, b):
    return a * np.power(x, -b)

def exponential(x, a, b):
    return a * np.exp(-b * x)

def stretched_exp(x, a, b, c):
    return a * np.exp(-b * np.power(x, c))

def logarithmic(x, a, b):
    return a - b * np.log(x)

def linear_fn(x, a, b):
    return a - b * x

FUNC_MODELS = {
    'Power law':     {'func': power_law,     'p0': [1.0, 0.75], 'k': 2},
    'Exponential':   {'func': exponential,   'p0': [1.0, 0.05], 'k': 2},
    'Stretched exp': {'func': stretched_exp,  'p0': [1.0, 0.05, 0.5], 'k': 3},
    'Logarithmic':   {'func': logarithmic,   'p0': [1.0, 0.1],  'k': 2},
    'Linear':        {'func': linear_fn,     'p0': [1.0, 0.01], 'k': 2},
}

all_fits = {}
rows = []

for key, data in all_data.items():
    bc, bm = data['bc'], data['bm']
    n = len(bc)
    fits = {}
    for name, spec in FUNC_MODELS.items():
        try:
            bounds = (0, np.inf) if name != 'Logarithmic' else (-np.inf, np.inf)
            popt, _ = optimize.curve_fit(spec['func'], bc, bm, p0=spec['p0'],
                                         maxfev=10000, bounds=bounds)
            y_pred = spec['func'](bc, *popt)
            rss = np.sum((bm - y_pred) ** 2)
            tss = np.sum((bm - np.mean(bm)) ** 2)
            r2 = 1 - rss / tss if tss > 0 else 0
            aic = n * np.log(rss / n) + 2 * spec['k'] if rss > 0 else np.inf
            bic = n * np.log(rss / n) + spec['k'] * np.log(n) if rss > 0 else np.inf
            fits[name] = {'r2': r2, 'aic': aic, 'bic': bic, 'params': popt}
        except:
            pass
    all_fits[key] = fits
    
    if fits:
        best_aic = min(fits, key=lambda k: fits[k]['aic'])
        best_bic = min(fits, key=lambda k: fits[k]['bic'])
        best_aic_val = fits[best_aic]['aic']
        for name, f in fits.items():
            rows.append({
                'dataset': key, 'label': data['label'], 'model': name,
                'r2': f['r2'], 'aic': f['aic'], 'bic': f['bic'],
                'delta_aic': f['aic'] - best_aic_val,
                'best_aic': name == best_aic, 'best_bic': name == best_bic,
            })

print('Fitting complete')

In [ ]:
# === Results table ===
print(f'{"Dataset":<30} {"Best AIC":<15} {"PL R²":>8} {"Exp R²":>8} {"ΔAIC(PL-Exp)":>14}')
print('-' * 78)

aic_winners = []
bic_winners = []

for key, data in all_data.items():
    fits = all_fits[key]
    if not fits:
        continue
    best_aic = min(fits, key=lambda k: fits[k]['aic'])
    best_bic = min(fits, key=lambda k: fits[k]['bic'])
    aic_winners.append(best_aic)
    bic_winners.append(best_bic)
    
    pl_r2 = fits.get('Power law', {}).get('r2', 0)
    exp_r2 = fits.get('Exponential', {}).get('r2', 0)
    pl_aic = fits.get('Power law', {}).get('aic', np.inf)
    exp_aic = fits.get('Exponential', {}).get('aic', np.inf)
    delta = pl_aic - exp_aic
    
    print(f'{data["label"]:<30} {best_aic:<15} {pl_r2:>8.4f} {exp_r2:>8.4f} {delta:>14.2f}')

print(f'\nAIC winners ({len(aic_winners)} datasets):')
for model, count in Counter(aic_winners).most_common():
    print(f'  {model:<20}: {count}/{len(aic_winners)} ({count/len(aic_winners)*100:.0f}%)')

print(f'\nBIC winners ({len(bic_winners)} datasets):')
for model, count in Counter(bic_winners).most_common():
    print(f'  {model:<20}: {count}/{len(bic_winners)} ({count/len(bic_winners)*100:.0f}%)')

In [ ]:
# === Per-dataset detailed breakdown ===
for key, data in all_data.items():
    fits = all_fits[key]
    if not fits:
        continue
    print(f'\n--- {data["label"]} ---')
    print(f'  {"Model":<20} {"R²":>8} {"AIC":>10} {"ΔAIC":>8}')
    print(f'  {"-"*48}')
    best_aic_val = min(f['aic'] for f in fits.values())
    for name in ['Power law', 'Exponential', 'Stretched exp', 'Logarithmic', 'Linear']:
        if name in fits:
            f = fits[name]
            delta = f['aic'] - best_aic_val
            marker = ' ★' if delta == 0 else ''
            print(f'  {name:<20} {f["r2"]:>8.4f} {f["aic"]:>10.2f} {delta:>8.2f}{marker}')

In [ ]:
# === Mean R² and ΔAIC per functional form ===
import pandas as pd
df = pd.DataFrame(rows)

if len(df) > 0:
    print('Mean R² across all datasets:')
    for model, r2 in df.groupby('model')['r2'].mean().sort_values(ascending=False).items():
        print(f'  {model:<20}: {r2:.4f}')
    
    print('\nMean ΔAIC (lower = better):')
    for model, daic in df.groupby('model')['delta_aic'].mean().sort_values().items():
        print(f'  {model:<20}: {daic:.2f}')